In [ ]:
import yaml
import glob
import json
import os
from dataclasses import dataclass
from pprint import pprint
from typing import Dict, Optional, List, Any
from slideguard.schemes import FullEvaluation
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.language_models import LanguageModelInput


# Functions

In [2]:
class VLLMChatOpenAI(ChatOpenAI):
    def _get_request_payload(
        self,
        input_: LanguageModelInput,
        *,
        stop: Optional[List[str]] = None,
        **kwargs: Any,
    ) -> dict:
        payload = super()._get_request_payload(input_, stop=stop, **kwargs)
        # max_tokens was deprecated in favor of max_completion_tokens
        # in September 2024 release
        if "max_completion_tokens" in payload:
            payload["max_tokens"] = payload.pop("max_completion_tokens")
        return payload
    

def load_goldens(base_path: str = "../golden") -> Dict[str, dict]:
    golden_files = glob.glob(os.path.join(base_path, "*.yaml"))

    goldens = dict()
    for file in golden_files:
        deck_name, _ = os.path.splitext(os.path.basename(file))
        with open(file, "r") as f:
            evaluation = yaml.safe_load(f)
        goldens[deck_name] = evaluation
    
    return goldens


def load_evaluations(base_path: str = "../slidedecks_test_evaluations") -> Dict[str, FullEvaluation]:
    evaluation_files = glob.glob(os.path.join(base_path, "evaluations_*.json"))

    evaluations = dict()
    for file in evaluation_files:
        deck_name, _ = os.path.splitext(os.path.basename(file))
        deck_name = deck_name.replace("evaluations_", "")
        with open(file, "r") as f:
            evaluation = FullEvaluation.model_validate_json(f.read())
        evaluations[deck_name] = evaluation
    
    return evaluations

In [21]:

system_prompt = """
You are a helpful assistant.
You need to estimate if a golden comment made by a human expert is presented in the set of evaluation comments made by an AI agent. 
The expert's comment may have a different structure and form than the evaluation comment, 
but the essence of the comment should be the same.
AI agent may have many comments in the set, so you need to find at least one comment that is the most similar to the expert's comment.


Cite the closest comment from the set of evaluation comments if any.
And then answer with "yes" or "no" on a new line.
"""

human_prompt = """
The comment made by a human expert:
{human_comment}

The set of evaluations comment made by an AI agent:
{ai_comments}
"""
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", human_prompt)
])

llm = VLLMChatOpenAI(
    model="/model",
    temperature=0.1,
    max_completion_tokens=1000,
    max_tokens=1000,
    base_url="http://d.dgx:8082/v1",
    api_key="token-abc123"
)

chain = (
    chat_prompt | llm | StrOutputParser()
)

In [26]:
goldens = load_goldens()
evaluations = load_evaluations()

print("Goldens:", len(goldens))
print("Evaluations:", len(evaluations))

Goldens: 3
Evaluations: 3


In [27]:
golden2evaluation = dict()
for deck_name, golden in goldens.items():
    if deck_name in evaluations:
        golden2evaluation[deck_name] = (golden, evaluations[deck_name])
    else:
        print(f"No evaluation for {deck_name}")

print(len(golden2evaluation))

3


In [ ]:
print("Evaluating deck structure analysis")

gold, evaluation = golden2evaluation['1_EN_Kataeva_Thesis']

ev_results = dict()

def _convert(el):
    del el['severity']
    return el

for criteria, evaluations in evaluation.deck_evaluations.evaluations.items():
    eval_results = [_convert(el) for el in evaluations['evaluation_results'] if el['severity'] > 2]
    ev_results[criteria.value] = eval_results



Evaluating deck structure analysis
----------------------------------------------------------------------------------------------------
Expert comment: Отсутствует обзор литературы (альтернативные подходы)
Result: The closest comment from the set of evaluation comments is not present as none of the AI agent's comments directly address the absence of a literature review or alternative approaches.

no
----------------------------------------------------------------------------------------------------
Expert comment: Не хватает слайда с обзором конкурирующих решений
Result: No


In [30]:
pprint(ev_results[criteria])

[{'evaluation_element': 'Motivation → Goal Connection',
  'evaluation_suggestion': 'The connection between the motivation and the goal '
                           'could be clearer. While the motivation is implied '
                           'in the challenges of topic modeling evaluation, '
                           'the goal should explicitly address these '
                           'challenges. Consider restating the goal to '
                           'directly tackle the issues identified in the '
                           'current state.'},
 {'evaluation_element': 'Solution → Experiments Connection',
  'evaluation_suggestion': 'The experiments should thoroughly test the '
                           'proposed solution. Ensure that the experiment '
                           'settings and steps are designed to validate the '
                           'effectiveness of the new attention-based metrics '
                           'in comparison to existing ones.'},
 {'evaluat

In [29]:
criteria = 'deck_storytelling'
# pprint(ev_results['deck_structure_analysis'])
for comment in gold['evaluation']['deck'][criteria]:
    result = chain.invoke({
        "human_comment": comment,
        "ai_comments": str(ev_results[criteria])
    })
    print("-" * 100)
    print("Expert comment:", comment)
    print("Result:", result)

----------------------------------------------------------------------------------------------------
Expert comment: Неочевидно как цель и задачи связаны с проблемой метрик
Result: The closest comment from the set of evaluation comments is:

"The connection between the motivation and the goal could be clearer. While the motivation is implied in the challenges of topic modeling evaluation, the goal should explicitly address these challenges. Consider restating the goal to directly tackle the issues identified in the current state."

yes
----------------------------------------------------------------------------------------------------
Expert comment: Intro есть, но нет описание почему topic modelling важен и для чего нужен
Result: The closest comment from the set of evaluation comments is:

"The connection between the motivation and the goal could be clearer. While the motivation is implied in the challenges of topic modeling evaluation, the goal should explicitly address these challen